In [0]:
silver_sickness = spark.table('silver_sick_leave')
dim_time_gold = spark.table("date_dimension")
dim_region_gold = spark.table("region_dimension")

In [0]:
# absent rate change by previous month

from pyspark.sql.functions import col, lag, round
from pyspark.sql.window import Window


organisation_window = (
    Window
    .partitionBy("org_code")
    .orderBy("month")
)

#  calculate the measures before joining dimensions
sickness_measures = (
    silver_sickness

    .withColumn(
        "previous_absence_rate_pct",
        lag("sickness_absence_rate_pct").over(organisation_window)
    )

    .withColumn(
        "absence_rate_change_pp",
        round(
            col("sickness_absence_rate_pct")
            - col("previous_absence_rate_pct"),
            2
        )
    )
)

In [0]:
from pyspark.sql.functions import col, lag, round, row_number
from pyspark.sql.window import Window

row_window = (
Window.orderBy(
    col("date_key"),
    col("region_key")
    )
)

# One timeline for each region
region_window = (
    Window
    .partitionBy("region_name")
    .orderBy("month")
)


# Calculate sickness-absence measurements
sickness_measures = (
    silver_sickness


    .withColumn(
        "previous_absence_rate_pct",
        lag("sick_leave_rate").over(region_window)
    )

    .withColumn(
        "absence_rate_change_pp",
        round(
            col("sick_leave_rate")
            - col("previous_absence_rate_pct"),
            2
        )
    )
)


# Add the dimension keys
fact_region_sickness_absence_gold = (
    sickness_measures.alias("a")

    # Date
    .join(
        dim_time_gold.alias("t"),
        col("a.month") == col("t.date"),
        how="left"
    )

    # Region
    .join(
        dim_region_gold.alias("r"),
        col("a.region_name") == col("r.region_name"),
        how="left"
    )

    .withColumn(
        "absence_key",
        row_number().over(row_window)
    )

    .select(
        col("absence_key"),
        col("r.region_key"),
        col("t.date_key"),
        col("a.sick_leave_rate"),
        col("a.previous_absence_rate_pct"),
        col("a.absence_rate_change_pp")
    )
)

display(fact_region_sickness_absence_gold)

In [0]:
fact_region_sickness_absence_gold.where(col("region_key") == 3).show(10)

In [0]:
  (fact_region_sickness_absence_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "path",
        "abfss://gold@jdnhsbronze.dfs.core.windows.net/fact_sick_leave/"
    ) \
    .saveAsTable(
        "fact_sick_leave"
    ))